# Stage Sharding — Quickstart

**Read many-small-file datasets fast.**

If your training data is millions of small unstructured files on a Snowflake stage (images,
audio, …), reading them every epoch is slow — each file is
its own network round-trip. **Stage sharding** repacks those small files into a
few large `.arrow` shards **once**, while maintaining their bytes footprint to prevent data bloating.

This notebook walks through the whole flow end to end:

1. **Generate shards** (one-time repack). Only re-run if need to update data.
2. **Read** the shards back with a Snowflake Shard Ray datasource.
3. **Train** distributed on **GPU** (decode + train, one worker per GPU) with
   `ShardedDataConnector` + `PyTorchDistributor`.
4. **Tune** the one knob (`target_shard_size_mb`).

> Runs on Snowflake Notebooks on **Container Runtime** (the ML runtime), where an
> active Snowpark session and a Ray cluster are already available. Step 3 trains on
> GPU, so use a **GPU-backed** compute pool for that step. Swap the stage paths below
> for your own. The example uses images, but sharding works for **any** file type —
> you own the decode step.

## Setup

Grab the active Snowpark session. On Container Runtime this connects you to your
account automatically — no credentials needed in the notebook.

In [ ]:
from snowflake.snowpark import Session

from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Point these at your own stage: RAW holds your many small image files,
# SHARDS is where the repacked .arrow files will be written.
RAW_PATH = "@MY_DB.MY_SCHEMA.MY_STAGE/raw_images"
SHARDS_PATH = "@MY_DB.MY_SCHEMA.MY_STAGE/shards_images"
FILE_PATTERN = "*.jpg"  # keep unrelated files out of the shards

## Step 1 — Generate shards (one-time)

`generate_shards` reads the raw, still-compressed bytes of every matching file and
repacks them into a few large `.arrow` shards. Your bytes are stored **verbatim**
(a JPEG stays a JPEG — no re-encoding, no quality loss).

Run this **once** after your data lands (or whenever it changes). It's an offline,
one-time cost — not something your training loop pays every run.

> **One-time, full pass.** `generate_shards` reads *every* source file once and
> writes the shards — about the cost of one slow-path epoch. So the speedup only
> pays off across **multiple epochs / repeated reads**; if you will read the data
> just once, do not shard it.

In [ ]:
from snowflake.ml.ray.stage_sharding import generate_shards

generate_shards(
    RAW_PATH,
    SHARDS_PATH,
    file_pattern=FILE_PATTERN,
    target_shard_size_mb=256,  # 256 MB per shard (the default; see "Tuning" below)
    force_cleanup=True,         # clear any old .arrow shards for a clean replace
)

# For reference: on 100k JPEGs (~1.85 GB) this produced 8 shards in ~10 min.

# Sanity check: list what we produced.
session.sql(f"LIST {SHARDS_PATH}").show()

## Step 2 — Read the shards back

Every training run does this. One shard = one read task = one Ray block, so the
**number of shards is your read/decode parallelism**.

The trailing slash on the path (`.../shards_images/`) anchors the read to exactly
this folder — without it, listing is prefix-based and could also pick up a sibling
folder like `shards_images_v2`.

In [ ]:
import ray
from snowflake.ml.ray.datasource import SFStageShardDataSource

ds = ray.data.read_datasource(
    SFStageShardDataSource(stage_location=f"{SHARDS_PATH}/")
)

# Each row carries the raw file bytes in a `bytes` column, plus a `path` column
# with the original file path (from include_paths=True, the default).
print("schema:", ds.schema())
print("rows:  ", ds.count())

## Step 3 — Distributed GPU training with the PyTorch distributor

For real distributed training, wrap the sharded dataset in a **`ShardedDataConnector`**
and hand it to the **`PyTorchDistributor`**, which runs **one worker per GPU**. The
connector splits the data into one shard per worker; each worker pulls *its own* shard
inside the training function and iterates it over epochs. Three calls:

1. `ShardedDataConnector.from_ray_dataset(ds, ingestor_class=RayDataIngester)` — turn
   the Ray dataset into a shardable connector.
2. `PyTorchDistributor(train_func, scaling_config=...)` — declare the cluster shape
   (one GPU worker per GPU: `num_gpus=1`, `num_workers_per_node` = GPUs per node).
3. `distributor.run(dataset_map={"train": connector})` — each worker calls
   `get_context().get_dataset_map()["train"].get_shard().to_torch_dataset()` and trains
   on its GPU.

> The training function runs on **every GPU worker** and gets its own data shard from
> the context — you don't pass the dataset in through a closure. Ray Train sets up the
> process group and places each worker on one GPU (`cuda:local_rank`); DDP syncs
> gradients across them. Keep the imports **inside** `train_func` so it ships cleanly
> to the workers. (Falls back to CPU if no GPU is present — handy for a local smoke test.)

In [ ]:
import io
import numpy as np
from PIL import Image
from snowflake.ml.data.sharded_data_connector import ShardedDataConnector
from snowflake.ml.modeling.distributors.pytorch import (
    PyTorchDistributor,
    PyTorchScalingConfig,
    WorkerResourceConfig,
)
# The runtime's ingestor for Ray datasets — required for from_ray_dataset().
from implementations.ray_data_ingester import RayDataIngester


# Decode to a fixed-size image array + a label, then build a shardable connector
# straight from the Ray dataset. DEMO ONLY: we derive a placeholder label — in a
# real job the label comes from the file path (e.g. parent folder = class) or a
# joined table.
def decode_with_label(batch):
    import io
    import numpy as np
    from PIL import Image

    images, labels = [], []
    for raw, path in zip(batch["bytes"], batch["path"]):
        img = Image.open(io.BytesIO(raw)).convert("RGB").resize((64, 64))
        images.append(np.asarray(img, dtype=np.float32) / 255.0)
        labels.append(sum(path.encode("utf-8")) % 2)  # <-- replace with your label
    return {"image": np.stack(images), "label": np.array(labels, dtype=np.int64)}


train_ds = ds.map_batches(decode_with_label, batch_format="numpy")

# 1. Wrap the Ray dataset in a shardable connector (one shard per worker).
train_connector = ShardedDataConnector.from_ray_dataset(
    train_ds, ingestor_class=RayDataIngester
)


# 2. The training function — runs on every GPU worker.
def train_func():
    import torch
    from torch import nn
    from torch.nn.parallel import DistributedDataParallel as DDP
    from torch.utils.data import DataLoader
    import ray.train
    from snowflake.ml.modeling.distributors.pytorch import get_context

    ctx = get_context()
    local_rank = ctx.get_local_rank()
    use_cuda = torch.cuda.is_available()
    device = torch.device(f"cuda:{local_rank}" if use_cuda else "cpu")

    # This worker's shard of the data -> a torch dataset that yields column->tensor
    # dict batches. The shard does the batching; DataLoader just passes through.
    torch_ds = ctx.get_dataset_map()["train"].get_shard().to_torch_dataset(batch_size=2048)
    loader = DataLoader(torch_ds, batch_size=None)

    # A tiny model — swap in your own. Ray Train already initialized the process
    # group; the model lives on this worker's GPU and DDP syncs gradients across them.
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(64 * 64 * 3, 64),
        nn.ReLU(),
        nn.Linear(64, 2),
    ).to(device)
    model = DDP(model, device_ids=[local_rank]) if use_cuda else DDP(model)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(3):  # <-- the multi-epoch iteration over the shard
        model.train()
        seen, running = 0, 0.0
        for batch in loader:
            images = batch["image"].to(device)          # (N, 64, 64, 3)
            labels = batch["label"].flatten().to(device)  # (N, 1) -> (N,)
            preds = model(images)
            loss = loss_fn(preds, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            seen += labels.shape[0]
            running += loss.item()

        # Every worker must call report (it's a collective); rank-0 metrics surface
        # in the result. To persist the model, pass
        # checkpoint=ray.train.Checkpoint.from_directory(dir) on rank 0.
        ray.train.report({"epoch": epoch, "loss": running, "images_seen": seen})
        print(f"[rank {ctx.get_world_rank()}] epoch {epoch}: {seen} imgs, loss {running:.2f}")


### Launch the distributed run

With the connector and `train_func` defined above, declare the cluster shape — **one worker per GPU** — and start training. Set `num_workers_per_node` to your node's GPU count (or omit `scaling_config` to auto-detect the cluster).

In [ ]:
# 3. Declare the cluster shape and run — one worker per GPU. Set num_workers_per_node
#    to the number of GPUs on each node (e.g. 4 for a 4-GPU node) and num_nodes to how
#    many nodes to use. Omit scaling_config entirely to auto-detect the whole cluster.
distributor = PyTorchDistributor(
    train_func=train_func,
    scaling_config=PyTorchScalingConfig(
        num_nodes=1,
        num_workers_per_node=4,  # <-- set to GPUs-per-node
        resource_requirements_per_worker=WorkerResourceConfig(num_cpus=0, num_gpus=1),
    ),
)
result = distributor.run(dataset_map={"train": train_connector})
print("metrics:", result.get_metrics())


## Tuning the one knob: `target_shard_size_mb`

This notebook uses the default **256 MB** per shard. Shard size is the only sizing
control, and it's a simple trade-off:

- **Larger shards** (e.g. 256 MB): fewer files → less per-file overhead, but fewer parallel read/decode units.
- **Smaller shards** (e.g. 32–64 MB): more files → more parallel readers, at the cost of slightly more per-file overhead.

Each shard is read by exactly one worker at a time, so **the number of shards is your
read/decode parallelism**. Rule of thumb: pick a size that produces at least as many
shards as you have workers. For a ~2 GB dataset, `64 MB` → ~30 shards (lots of
parallelism); `256 MB` → ~8 shards (fine for a single node, thin for a big cluster).

## Notes & tips

- **Order is preserved.** Sharding never shuffles. Shuffle at read time with
  `decoded.random_shuffle()` or during iterating with `iter_batches(local_shuffle_buffer_size=...)`.
- **Re-running.** Writing again overwrites same-named shards; `force_cleanup=True`
  first clears old `.arrow` shards in the destination folder for a clean replace. Preventing data conflict on shard files path.
- **Read only some columns.** `SFStageShardDataSource(..., columns=["bytes"])` keeps
  just what you need.
- **Distributed training.** Feed the shards to the PyTorch distributor by wrapping
  them in `ShardedDataConnector.from_ray_dataset(ds, ingestor_class=RayDataIngester)` and passing
  `dataset_map={"train": connector}` to `PyTorchDistributor.run(...)` — see Step 3.
- **Pairs with the file cache.** Pass `use_file_cache=True` to also cache shards on
  node-local disk after the first read, speeding up later epochs even more.

**Rule of thumb:** if you have lots of small unstructured files and read them for multiple epocs,
shard them. In internal benchmarks that's a **~4.4–5.5× per-epoch read speedup** (measured on 100k JPEGs, 256 MB shards),
paying for itself in ~1.2–1.5 epochs.